# Data Cleaning

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)

### Load the data

In [5]:
df_raw = pd.read_parquet("data_for_student.parquet")
meta_raw = pd.read_parquet("metadata_for_student.parquet")
print("data    :", df_raw.shape)
print("metadata:", meta_raw.shape)

data    : (355532, 5)
metadata: (350066, 3)


### What does the meta data say
02097a_BG7-TUM_isoform_79_01_01-2xIT_2xHCD-1h-R3  
└─batch─┘ └──── pool ────┘       └─── method ───┘  

In absolute terms:
| Level | Distinct values |
|---|---|
| raw_file (run) | 964 |
| pool | 540 |
| batch (02097a, 01640c, …) | 20 |
| acquisition method | 10 |

#### What about leakage?
After merging data and metadata we get 350.176 unique modified peptide sequences, meaning that there are no replicate spectra, no peptides appearing at two charge states. The dataset has been pre-de-duplicated.

#### Residual overlap
After stripping `[UNIMOD:x]` there are 305.004 unique stripped backbones, lost ~45k rows -> 13% share a backbone with another row as modified variants.
--> Grouping on the stripped sequence is worh it: it prevents `PEPTIDEM` and `PEPTIDEM[UNIMOD:35]` from straddling the split.

#### Cross-batch overlap
Only 0.4% of stripped sequences span more than one batch, covering 0.8% of rows. No run-level holdout necessary. Pools are near-disjoint in peptide content by design, so a pool-level split would be almost identical to a sequence-grouped split.

### One Problem tho :3
There are two main acquisition regimes:
| Method | Rows |
|---|---|
| 2xIT_2xHCD-1h | 204,679 |
| DDA-1h | 145,382 |
| DDA-longexcl | 115 |

Upon inspecting a 40k sample these differ systematically:
| Method | mean peaks | mean length | y-ion intensity fraction | mean charge |
|---|---|---|---|---|
| 2xIT_2xHCD-1h | 19.70 | 13.33 | 0.656 | 2.38 |
| DDA-1h | 19.99 | 13.95 | 0.633 | 2.48 |

especially on charge composition:
| Method | z=2 | z=3 | z=4 |
|---|---|---|---|
| 2xIT_2xHCD-1h | 65.3% | 31.8% | 2.9% |
| DDA-1h | 55.2% | 41.2% | 3.5% |
  

This can lead to precursor_charge bein confounded with acquisition method. When a model is trained on sequence + charge it could potentially learn method-specific bias through the charge feature.  
Three solutions:
1. Stratify on method x charge -> same acquisition mixture
2. Report test metrices on Spectral Angle x charge -> catch regressions earlier, tell if a single moddle is enough.
3. Method as input feature. Prosit does it (they condition on collision energy but we don't have NCE values, but the method token could be a proxy). But I don't like it, it does not generalize well TODO: @Dana what do we doe

Final verdict:
Split on method x charge, drop `DDA-longexcl` as they are a distinct class with too few examples to model or evaluate 

### Meta about names
The names say `HCD` and `DDA`, not `CID`. ProteomeTools acquires both CID and HCD scasns per precursor so its plausible that CID subset
TODO @Fridolin ask supervisor  $namehiereinfuegen

# Data cleaning time


## Step 1 - Merge and decompose acquisition identifier
Remember that:    
02097a_BG7-TUM_isoform_79_01_01-2xIT_2xHCD-1h-R3           
└batch┘     └──── pool ────┘     └─── method ───┘

In [ ]:
df = df_raw.merge(meta_raw, on=["raw_file", "scan_number"], how="inner")
print(f"spectra   : {len(df_raw):>7,}")
print(f"metadata  : {len(meta_raw):>7,}")
print(f"merged    : {len(df):>7,}  (dropped {len(df_raw) - len(df):,} spectra without precursor charge)")

# everything after the first hyphen is "<pool>_<fraction>_<rep>-<method>"
rest = df["raw_file"].str.split("-", n=1).str[1]

df["batch"] = df["raw_file"].str.extract(r"^(\w{6})_")
df["pool"] = rest.str.replace(r"_\d\d_\d\d-.*$", "", regex=True)
df["method"] = rest.str.extract(r"_\d\d_\d\d-(.*)$")

# The method token is further split into the acquisition regime (frag, e.g. `2xIT_2xHCD-1h`) and the technical replicate suffix (R1 .. R5)
df["frag"] = df["method"].str.replace(r"-R\d$", "", regex=True)
df["replicate"] = df["method"].str.extract(r"-(R\d)$")

# UNIMOD stripping yields the aa backbone
df["stripped_sequence"] = df["peptide_sequence"].str.replace(r"\[UNIMOD:\d+\]", "", regex=True)
df["sequence_length"] = df["stripped_sequence"].str.len()

print()
for col in ["batch", "pool", "raw_file", "method", "frag", "replicate"]:
    print(f"{col:<12}: {df[col].nunique():>5,} distinct")

## Step 2 - Discriminate on acquisition regime and precursor charge
Aint I a edgy fellah lol

Reasoning:
Acquisition regime: `DDA-longexcl` constitues a distinct experimental setup, which completely goes out of bounds of the other regimes mean peptide length of 25.9 vs 13.5 elsewhere and 84% tripluy charged vs rouglhy 35% elsewhere withonly 115 bum ass spectra. Insufficient either to fit regime specific model behaviour or to measure it, so we can kick them out as not to dilute the majority regimes.

Precursor charge: Distribution imbalanced; Charge 1 (52 spectra), Charge 5 (217), Charge 6 (1) cannot suppiort a meaningful held out evaluatoni: we restrict analysis to charge 2-4, which is also conveniently the range modelled in the CID fragment intensity literature [citation needed].  
This excludes a total of 270 spectra or 0.08% of the data

In [7]:
KEEP_CHARGES = (2, 4)
DROP_FRAG = ["DDA-longexcl"]

n_start = len(df)
mask_frag = ~df["frag"].isin(DROP_FRAG)
mask_charge = df["precursor_charge"].between(*KEEP_CHARGES)

print("excluded acquisition regimes:")
print(df.loc[~mask_frag, "frag"].value_counts().to_string() or "  none")
print("\nexcluded charge states:")
print(df.loc[~mask_charge, "precursor_charge"].value_counts().sort_index().to_string() or "  none")

df = df[mask_frag & mask_charge].reset_index(drop=True)
print(f"\nretained  : {len(df):,} / {n_start:,}  ({len(df) / n_start:.2%})")


excluded acquisition regimes:
frag
DDA-longexcl    115

excluded charge states:
precursor_charge
1     52
5    217
6      1

retained  : 349,791 / 350,176  (99.89%)


## Step 3 - Grouped, stratified partition

A partition must satisfy two requirements: Grouping and stratification.

Grouping: Although the dataset is deduplicated at the modified sequence level, it is not when stripped of modifications. 13% of rows share a bare aa backbone. As a peptide and its oxidised or carbamidomethylated counterpart produce highly correlated fragment intensity patterns we have to make sure to contain them to either train/test/validation dataset as not to leak information from training into evaluation.  
Therefore we group on the stripped backbone.

Stratification: As `precursor_charge` is confounded with acquisition method (`2xIT_2xHCD-1h` runs are 65.3% doubly charged against 55.2% for `DDA-1h`) we cannot stratify on charge alone as that would permit the regime mixture to drift between training and test.  
Therefore we stratificate on `frag x precursor_charge`. Peptide length is not included in the label as fragment count scales with length and is implied. Length balance will be verified later.

Methodology: `StratifiedGroupKFold` does both and  `n_splits=10` yields us a 80/10/10 train/val/test dataset. Fixing `random_state` for reproducible runs

In [15]:
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_STATE = 187
N_SPLITS = 10  # 10 folds -> 80:10:10 train:val:test

strat = df["frag"] + "_z" + df["precursor_charge"].astype(str)
groups = df["stripped_sequence"]

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = [test_idx for _, test_idx in sgkf.split(df, strat, groups)]

test_idx = folds[0]
val_idx = folds[1]
train_idx = np.hstack(folds[2:])

df["split"] = "train"
df.loc[val_idx, "split"] = "val"
df.loc[test_idx, "split"] = "test"

print(df["split"].value_counts(normalize=True))


split
train    0.799998
test     0.100003
val      0.100000
Name: proportion, dtype: float64


## Step 4 - Verification 
We check for
1. Disjoint backbones: No stripped backbone may be in more then one dataset
2. Stratum balance: Marginal distribution and precursor change should agree between all dataset within tolerance
3. Length balance: We did not straitifacte on it, here we have to confirm through backbone grouping

In [20]:
tr, val, te = df[df.split == "train"], df[df.split == "test"], df[df.split == "val"]

# Disjoint backbones
assert len(pd.merge(tr, val, on="stripped_sequence", how="inner")) == 0, "Leakage between Train and Val!"
assert len(pd.merge(tr, te, on="stripped_sequence", how="inner")) == 0, "Leakage between Train and Test!"
assert len(pd.merge(te, val, on="stripped_sequence", how="inner")) == 0, "Leakage between Test and Val!"

print(f"[ok] no backbone shared between train, val, test")

# Review stratum balance by hand
print("\n ==== stratum balance ==== ")
for col in ["frag", "precursor_charge"]:
    comp = pd.DataFrame({
        "train_%": tr[col].value_counts(normalize=True) * 100,
        "val_%": val[col].value_counts(normalize=True) * 100,
        "test_%": te[col].value_counts(normalize=True) * 100,
    })

    # Calculate maximum peak to peak 
    pct_cols = ["train_%", "val_%", "test_%"]
    comp["delta_pp"] = comp[pct_cols].max(axis=1) - comp[pct_cols].min(axis=1)

    print(f"\n{col}\n{comp.round(2).to_string()}")

# Review length balance by hand
print("\n ==== length balance ==== ")
length = pd.DataFrame({
    "train": tr.sequence_length.describe(), 
    "val": val.sequence_length.describe(),  # Added validation split
    "test": te.sequence_length.describe()
})
print(f"\nsequence_length\n{length.round(2).to_string()}")



[ok] no backbone shared between train, val, test

 ==== stratum balance ==== 

frag
               train_%  val_%  test_%  delta_pp
frag                                           
2xIT_2xHCD-1h    58.49  58.49   58.49       0.0
DDA-1h           41.51  41.51   41.51       0.0

precursor_charge
                  train_%  val_%  test_%  delta_pp
precursor_charge                                  
2                   61.13  61.13   61.13       0.0
3                   35.75  35.75   35.75       0.0
4                    3.12   3.12    3.12       0.0

 ==== length balance ==== 

sequence_length
           train       val      test
count  279832.00  34980.00  34979.00
mean       13.59     13.59     13.59
std         4.70      4.67      4.71
min         7.00      7.00      7.00
25%        10.00     10.00     10.00
50%        13.00     13.00     13.00
75%        16.00     16.00     16.00
max        30.00     30.00     30.00


## Step 6 - Save to disk

In [ ]:
OUT = "clean_spectra_split.parquet"
trainout = OUT.replace("_split", "_train_split")
testout = OUT.replace("_split", "_test_split")
valout = OUT.replace("_split", "_val_split")

df.to_parquet(OUT, index=False)

# create convenience datasets
df[df.split == "train"].to_parquet(trainout, index=False)
df[df.split == "test"].to_parquet(testout, index=False)
df[df.split == "val"].to_parquet(valout, index=False)

print(f"wrote {OUT}: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(df.columns.tolist())

df.head(5)

wrote clean_spectra_split.parquet: 349,791 rows x 14 cols
['raw_file', 'scan_number', 'peptide_sequence', 'intensities_raw', 'matched_ions', 'precursor_charge', 'batch', 'pool', 'method', 'frag', 'replicate', 'stripped_sequence', 'sequence_length', 'split']


,raw_file,scan_number,peptide_sequence,intensities_raw,matched_ions,precursor_charge,batch,pool,method,frag,replicate,stripped_sequence,sequence_length,split
0,01650b_BG4-TUM_first_pool_73_01_01-2xIT_2xHCD-...,48156,HLTVEELFGTSLPK,"[116.713806152344, 54029.1484375, 1521.8664550...","[y1, y2, b2, b3, y3, y4, b4, y5, b5, y6, b6, y...",3,01650b,TUM_first_pool_73,2xIT_2xHCD-1h-R2,2xIT_2xHCD-1h,R2,HLTVEELFGTSLPK,14,test
1,02208a_GC10-TUM_second_addon_34_01_01-2xIT_2xH...,5465,SM[UNIMOD:35]ASNHETAHNVIC[UNIMOD:4]K,"[10120.6884765625, 133350.921875, 22052.142578...","[y1, b2, b3, y2, b4, y3, b5, y4, y5, b6, y6, y...",4,02208a,TUM_second_addon_34,2xIT_2xHCD-1h-R1,2xIT_2xHCD-1h,R1,SMASNHETAHNVICK,15,train
2,02080d_GB1-TUM_isoform_109_01_01-2xIT_2xHCD-1h-R1,29548,PHKC[UNIMOD:4]PDC[UNIMOD:4]DMAFVTSGELVR,"[11643.265625, 12050.8876953125, 48106.8476562...","[y1, b2, y2, b3, y3, y4, b4, y5, b5, y6, b6, y...",4,02080d,TUM_isoform_109,2xIT_2xHCD-1h-R1,2xIT_2xHCD-1h,R1,PHKCPDCDMAFVTSGELVR,19,train
3,02097a_BF10-TUM_isoform_70_01_01-2xIT_2xHCD-1h-R3,53980,KQPALDVLYDVMK,"[1181.60241699219, 475.930603027344, 450.20574...","[b2, y2, b3, y3, b4, y4, b5, b6, y5, b7, y6, b...",2,02097a,TUM_isoform_70,2xIT_2xHCD-1h-R3,2xIT_2xHCD-1h,R3,KQPALDVLYDVMK,13,test
4,02080d_GA9-TUM_isoform_105_01_01-DDA-1h-R1,26218,HPGHIHKLLAQQLVSPVK,"[8962.16015625, 68957.0, 6599.31884765625, 827...","[y1, b2, y2, b3, y3, b4, y4, y5, b5, y6, b6, y...",4,02080d,TUM_isoform_105,DDA-1h-R1,DDA-1h,R1,HPGHIHKLLAQQLVSPVK,18,val


## Step 7 - Retain without precursor charge to enrich training


In [12]:
OUT_NO_CHARGE = "data_without_precursor_charge.parquet"

flagged = df_raw.merge(
    meta_raw[["raw_file", "scan_number"]].drop_duplicates(),
    on=["raw_file", "scan_number"],
    how="left",
    indicator=True,
)
no_charge = flagged[flagged["_merge"] == "left_only"].drop(columns="_merge").reset_index(drop=True)
print(f"spectra without metadata: {len(no_charge):,}")

rest_nc = no_charge["raw_file"].str.split("-", n=1).str[1]
no_charge["precursor_charge"] = pd.Series(pd.NA, index=no_charge.index, dtype="Int64")
no_charge["batch"] = no_charge["raw_file"].str.extract(r"^(\w{6})_")
no_charge["pool"] = rest_nc.str.replace(r"_\d\d_\d\d-.*$", "", regex=True)
no_charge["method"] = rest_nc.str.extract(r"_\d\d_\d\d-(.*)$")
no_charge["frag"] = no_charge["method"].str.replace(r"-R\d$", "", regex=True)
no_charge["replicate"] = no_charge["method"].str.extract(r"-(R\d)$")
no_charge["stripped_sequence"] = no_charge["peptide_sequence"].str.replace(r"\[UNIMOD:\d+\]", "", regex=True)
no_charge["sequence_length"] = no_charge["stripped_sequence"].str.len()
no_charge["split"] = pd.NA

print("\nexcluded acquisition regimes:")
print(no_charge.loc[no_charge.frag.isin(DROP_FRAG), "frag"].value_counts().to_string() or "  none")
no_charge = no_charge[~no_charge["frag"].isin(DROP_FRAG)].reset_index(drop=True)

no_charge = no_charge[df.columns]
no_charge.to_parquet(OUT_NO_CHARGE, index=False)
print(f"\nwrote {OUT_NO_CHARGE}: {no_charge.shape[0]:,} rows x {no_charge.shape[1]} cols")
print(f"\nfrag distribution:\n{no_charge.frag.value_counts().to_string()}")


spectra without metadata: 5,356

excluded acquisition regimes:
frag
DDA-longexcl    2

wrote data_without_precursor_charge.parquet: 5,354 rows x 14 cols

frag distribution:
frag
DDA-1h                 3414
2xIT_2xHCD-1hnoincl     991
DDA-1hnoincl            635
2xIT_2xHCD-1h           314
